# JanoGPT Training on Kaggle TPU v5e-8

This notebook trains a GPT-2 model using JanoGPT on Kaggle's TPU v5e-8.

**What this notebook does:**
1. Downloads JanoGPT code from GitHub
2. Installs TPU-compatible JAX
3. Sets up WandB for tracking
4. Downloads OpenWebText dataset
5. Trains GPT-2 124M for 1000 steps
6. **Uploads checkpoints to Google Drive every 100 steps**

**Requirements:**
- Enable TPU v5e-8 in Kaggle (Settings → Accelerator → TPU v5e-8)
- Add WandB API key as Kaggle secret (optional)
- Mount Google Drive for checkpoint uploads

**Training time:** ~20-30 minutes on TPU v5e-8 (much faster than GPU!)

**TPU v5e-8 specs:**
- 8 TPU cores
- 32GB HBM per core (256GB total)
- ~10x faster than T4 GPU for large models

## 1. Setup TPU Environment

In [ ]:
# Check TPU availability
import os
print(f"TPU_NAME: {os.environ.get('TPU_NAME', 'Not found')}")
print(f"COLAB_TPU_ADDR: {os.environ.get('COLAB_TPU_ADDR', 'Not found')}")

In [ ]:
# Clone JanoGPT repository
!git clone https://github.com/hhe0u0/janogpt.git
%cd janogpt

In [ ]:
# Install TPU-compatible JAX (specific version for TPU v5e)
!pip install -q jax[tpu] -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

In [ ]:
# Install other dependencies
!pip install -q flax optax orbax-checkpoint tiktoken tqdm numpy

In [ ]:
# Install WandB for experiment tracking (optional)
!pip install -q wandb

In [ ]:
# Verify JAX sees the TPU
import jax
print(f"JAX devices: {jax.devices()}")
print(f"Device count: {jax.local_device_count()}")
print(f"Device type: {jax.devices()[0].platform}")
print(f"\nExpected: 8 TPU cores")

if jax.devices()[0].platform != 'tpu':
    print("\n⚠️  WARNING: TPU not detected!")
    print("Make sure you enabled TPU v5e-8 in Kaggle settings")
else:
    print("\n✓ TPU detected successfully!")

## 2. Mount Google Drive for Checkpoint Uploads

**Important:** Kaggle sessions are temporary. Mount Google Drive so checkpoints persist beyond the session.

In [ ]:
# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/janogpt_checkpoints/kaggle_tpu"
    drive_available = True
    print(f"✓ Google Drive mounted")
    print(f"✓ Checkpoints will be saved to: {DRIVE_CHECKPOINT_DIR}")
except ImportError:
    print("⚠️  Google Colab not available (Kaggle environment)")
    print("Trying alternative method for Kaggle...")
    
    # For Kaggle, we'll use local storage and create zip files
    # User can download them from the output panel
    DRIVE_CHECKPOINT_DIR = None
    drive_available = False
    print("⚠️  Google Drive not available")
    print("Checkpoints will be saved locally. Download them from Kaggle output panel.")

## 3. Setup WandB (Optional)

In [ ]:
import os

# Configuration
wandb_enabled = True  # Set to False to disable WandB

if wandb_enabled:
    try:
        # Try to get WandB key from Kaggle secrets
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        wandb_key = user_secrets.get_secret("WANDB_API_KEY")
        os.environ["WANDB_API_KEY"] = wandb_key
        print("✓ Loaded WandB API key from Kaggle secrets")
    except:
        print("⚠️  WandB secret not found. Trying interactive login...")
        import wandb
        wandb.login()
else:
    print("ℹ️  WandB tracking disabled")

## 4. Setup Data

In [ ]:
# Check if OpenWebText dataset is available
from pathlib import Path
import os

# Kaggle dataset path
DATA_DIR = "/kaggle/input/datasets/windmaple/openwebtext-gpt2"
data_dir = Path(DATA_DIR)

if data_dir.exists():
    train_bin = data_dir / "train.bin"
    val_bin = data_dir / "val.bin"
    
    if train_bin.exists() and val_bin.exists():
        print(f"✓ Found OpenWebText dataset at {DATA_DIR}")
        print(f"  train.bin: {train_bin.stat().st_size / 1e9:.2f} GB")
        print(f"  val.bin: {val_bin.stat().st_size / 1e6:.2f} MB")
    else:
        print(f"⚠️  Dataset path exists but files not found")
else:
    print(f"⚠️  Dataset not found at {DATA_DIR}")
    print("\nAdd the dataset: Kaggle → Add data → Search 'openwebtext-gpt2'")

In [ ]:
# Create symbolic links to data
from pathlib import Path
import os

DATA_DIR = "/kaggle/input/datasets/windmaple/openwebtext-gpt2"
data_dir = Path(DATA_DIR)

# Create work directory
work_data_dir = Path("data/openwebtext")
work_data_dir.mkdir(parents=True, exist_ok=True)

if data_dir.exists():
    train_bin = data_dir / "train.bin"
    val_bin = data_dir / "val.bin"
    
    # Create symbolic links
    if not (work_data_dir / "train.bin").exists():
        os.symlink(train_bin, work_data_dir / "train.bin")
    if not (work_data_dir / "val.bin").exists():
        os.symlink(val_bin, work_data_dir / "val.bin")
    
    print(f"✓ Data linked to {work_data_dir}")
else:
    print(f"⚠️  Creating dummy data for testing...")
    import numpy as np
    
    dummy_train = np.random.randint(0, 50257, size=5_000_000, dtype=np.uint16)
    dummy_val = np.random.randint(0, 50257, size=500_000, dtype=np.uint16)
    
    dummy_train.tofile(work_data_dir / "train.bin")
    dummy_val.tofile(work_data_dir / "val.bin")
    
    print(f"✓ Created dummy dataset (for testing only)")

## 5. Create TPU-Optimized Training Configuration

**TPU v5e-8 optimizations:**
- `micro_batch_size: 8` - TPUs love large batches
- `gradient_accumulation_steps: 8` - Lower than GPU (TPU cores are fast)
- `num_devices: 8` - All 8 TPU cores
- Effective batch: 8 × 8 × 8 × 1024 = **524,288 tokens ≈ 0.5M** ✓
- Checkpoint every 100 steps → uploads to Google Drive

In [ ]:
import json
from pathlib import Path

# TPU v5e-8 optimized configuration
config = {
    "_comment": "GPT-2 124M training config for Kaggle TPU v5e-8 (1000 steps)",
    
    "model": {
        "dropout_prob": 0.1,
        "num_blocks": 12,
        "emb_dim": 768,
        "num_heads": 12,
        "seq_len": 1024,
        "epsilon": 1e-6,
        "voc_size": 50304  # Padded for efficiency
    },
    
    "optimizer": {
        "learning_rate": 6e-4,
        "min_learning_rate": 6e-5,
        "warmup_steps": 100,
        "beta1": 0.9,
        "beta2": 0.95,
        "grad_clip": 1.0,
        "weight_decay": 0.1
    },
    
    "training": {
        "max_steps": 1000,
        "micro_batch_size": 8,  # TPUs love large batches
        "gradient_accumulation_steps": 8,  # Lower than GPU (TPU is fast)
        "num_devices": 8,  # All 8 TPU cores
        "_effective_batch_comment": "8 × 8 × 8 × 1024 = 524,288 tokens ≈ 0.5M",
        "seed": 42
    },
    
    "data": {
        "data_dir": "data/openwebtext",
        "train_file": "train.bin",
        "val_file": "val.bin"
    },
    
    "logging": {
        "eval_interval": 100,
        "eval_iters": 50,
        "log_interval": 10
    },
    
    "checkpointing": {
        "save_interval": 100,  # Save every 100 steps (will upload to Drive)
        "output_dir": "output_kaggle_tpu",
        "resume_from_checkpoint": None
    },
    
    "wandb": {
        "enabled": wandb_enabled,
        "project": "janogpt-kaggle",
        "run_name": "gpt2-124m-tpu-v5e-8",
        "tags": ["kaggle", "gpt2", "tpu-v5e-8", "1000-steps"]
    }
}

# Save config
config_dir = Path("configs")
config_dir.mkdir(exist_ok=True)
config_path = config_dir / "train_kaggle_tpu.json"

with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"✓ Config saved to {config_path}")
print("\nTPU v5e-8 Configuration:")
print(f"  Model: GPT-2 124M ({config['model']['num_blocks']} layers, {config['model']['emb_dim']} dim)")
print(f"  TPU cores: {config['training']['num_devices']}")
print(f"  Micro batch per core: {config['training']['micro_batch_size']}")
print(f"  Gradient accumulation: {config['training']['gradient_accumulation_steps']}")
tokens_per_step = (config['training']['micro_batch_size'] * 
                   config['training']['gradient_accumulation_steps'] * 
                   config['training']['num_devices'] * 
                   config['model']['seq_len'])
print(f"  Effective batch: {tokens_per_step / 1000:.0f}K tokens per step")
print(f"  Checkpoint interval: every {config['checkpointing']['save_interval']} steps")
print(f"  Total steps: {config['training']['max_steps']}")
print(f"  WandB: {'Enabled' if config['wandb']['enabled'] else 'Disabled'}")
print(f"\n  Expected checkpoints: {config['training']['max_steps'] // config['checkpointing']['save_interval']} checkpoints")
print(f"  Storage needed: ~{(config['training']['max_steps'] // config['checkpointing']['save_interval']) * 0.5:.1f} GB")

## 6. Create Checkpoint Upload Helper

This script monitors for new checkpoints and uploads them to Google Drive automatically.

In [ ]:
%%writefile checkpoint_uploader.py
"""Background checkpoint uploader for Google Drive."""
import time
import shutil
from pathlib import Path
import sys

def upload_checkpoints(local_dir, drive_dir, interval=30):
    """
    Monitor local checkpoint directory and upload new checkpoints to Drive.
    
    Args:
        local_dir: Local checkpoint directory
        drive_dir: Google Drive checkpoint directory
        interval: Check interval in seconds
    """
    local_dir = Path(local_dir)
    drive_dir = Path(drive_dir)
    drive_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"Monitoring: {local_dir}")
    print(f"Uploading to: {drive_dir}")
    print(f"Check interval: {interval}s")
    print()
    
    uploaded = set()
    
    while True:
        try:
            if local_dir.exists():
                checkpoints = sorted(local_dir.glob("step_*"))
                
                for ckpt in checkpoints:
                    if ckpt.name not in uploaded:
                        dest = drive_dir / ckpt.name
                        
                        if dest.exists():
                            print(f"[{time.strftime('%H:%M:%S')}] {ckpt.name}: Already in Drive")
                            uploaded.add(ckpt.name)
                        else:
                            print(f"[{time.strftime('%H:%M:%S')}] {ckpt.name}: Uploading...", end="", flush=True)
                            start = time.time()
                            shutil.copytree(ckpt, dest)
                            elapsed = time.time() - start
                            size_mb = sum(f.stat().st_size for f in dest.rglob("*") if f.is_file()) / 1e6
                            print(f" ✓ ({size_mb:.1f} MB, {elapsed:.1f}s)")
                            uploaded.add(ckpt.name)
            
            time.sleep(interval)
            
        except KeyboardInterrupt:
            print("\nCheckpoint uploader stopped")
            break
        except Exception as e:
            print(f"Error: {e}")
            time.sleep(interval)

if __name__ == "__main__":
    local_dir = sys.argv[1] if len(sys.argv) > 1 else "output_kaggle_tpu/checkpoints"
    drive_dir = sys.argv[2] if len(sys.argv) > 2 else "/content/drive/MyDrive/janogpt_checkpoints/kaggle_tpu"
    upload_checkpoints(local_dir, drive_dir)

## 7. Start Background Checkpoint Uploader

This runs in the background and uploads checkpoints to Google Drive as they're created.

In [ ]:
import subprocess
import time

if drive_available and DRIVE_CHECKPOINT_DIR:
    print("Starting background checkpoint uploader...")
    
    # Start uploader in background
    uploader_process = subprocess.Popen(
        ["python", "checkpoint_uploader.py", 
         "output_kaggle_tpu/checkpoints", 
         DRIVE_CHECKPOINT_DIR],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    
    print(f"✓ Checkpoint uploader started (PID: {uploader_process.pid})")
    print(f"  Monitoring: output_kaggle_tpu/checkpoints")
    print(f"  Uploading to: {DRIVE_CHECKPOINT_DIR}")
    print()
    time.sleep(2)
else:
    print("⚠️  Google Drive not available, skipping background uploader")
    print("Checkpoints will be saved locally only")
    uploader_process = None

## 8. Train the Model

Now we'll train for 1000 steps. This should take about 20-30 minutes on TPU v5e-8.

**Expected TPU performance:**
- Tokens/sec: ~10,000-15,000 (10x faster than GPU!)
- Steps/sec: ~5-10
- Memory usage: ~8-12GB per TPU core

**Checkpoints:**
- Saved every 100 steps locally
- Automatically uploaded to Google Drive (if mounted)
- Total: 10 checkpoints (~5GB total)

In [ ]:
# Start training
!python scripts/train.py --config configs/train_kaggle_tpu.json --verbose

## 9. Stop Background Uploader

In [ ]:
if uploader_process is not None:
    print("Stopping checkpoint uploader...")
    uploader_process.terminate()
    uploader_process.wait(timeout=5)
    print("✓ Checkpoint uploader stopped")

## 10. Verify Checkpoints

In [ ]:
# List local checkpoints
from pathlib import Path

checkpoint_dir = Path("output_kaggle_tpu/checkpoints")

if checkpoint_dir.exists():
    checkpoints = sorted(checkpoint_dir.glob("step_*"))
    print(f"✓ Found {len(checkpoints)} local checkpoint(s):")
    for ckpt in checkpoints:
        total_size = sum(f.stat().st_size for f in ckpt.rglob("*") if f.is_file())
        size_mb = total_size / 1e6
        print(f"  {ckpt.name}: {size_mb:.1f} MB")
else:
    print("⚠️  No local checkpoints found")

print()

# List Drive checkpoints
if drive_available and DRIVE_CHECKPOINT_DIR:
    drive_dir = Path(DRIVE_CHECKPOINT_DIR)
    if drive_dir.exists():
        drive_checkpoints = sorted(drive_dir.glob("step_*"))
        print(f"✓ Found {len(drive_checkpoints)} Drive checkpoint(s):")
        for ckpt in drive_checkpoints:
            total_size = sum(f.stat().st_size for f in ckpt.rglob("*") if f.is_file())
            size_mb = total_size / 1e6
            print(f"  {ckpt.name}: {size_mb:.1f} MB")
    else:
        print("⚠️  No Drive checkpoints found")
else:
    print("ℹ️  Google Drive not mounted")

## 11. Test Text Generation

In [ ]:
# Find the latest checkpoint
checkpoint_dir = Path("output_kaggle_tpu/checkpoints")
checkpoints = sorted(checkpoint_dir.glob("step_*"))

if checkpoints:
    latest_checkpoint = checkpoints[-1]
    print(f"Testing with checkpoint: {latest_checkpoint}")
    
    # Generate text
    !python scripts/generate.py \
        --checkpoint {latest_checkpoint} \
        --prompt "Once upon a time" \
        --max_tokens 100 \
        --temperature 0.8
else:
    print("No checkpoint found to test")

## 12. Summary

**Training complete!**

Your checkpoints are:
- ✅ Saved locally: `output_kaggle_tpu/checkpoints/`
- ✅ Uploaded to Drive: `{DRIVE_CHECKPOINT_DIR}/` (if mounted)

**To resume training later:**
1. Mount Google Drive
2. Copy checkpoint from Drive to local
3. Update config: `resume_from_checkpoint: "path/to/checkpoint"`
4. Run training script

**To download checkpoints:**
- From Drive: Access from any device
- From Kaggle: Download from output panel

**TPU v5e-8 Performance Summary:**
- ~10x faster than T4 GPU
- ~5-10 steps/sec
- ~10,000-15,000 tokens/sec
- Checkpoint every 100 steps = ~5-10 minutes between saves